In [68]:
import pandas as pd

df = pd.read_csv("../data/sentimentos.csv", encoding="utf-8")

print(df)

                                texto  sentimento
0           super material resistente           1
1          realmente produto incrível           1
2        bastante péssima experiência           0
3         muito atendimento excelente           1
4                Dinheiro jogado fora           0
..                                ...         ...
495   muito chegou em perfeito estado           1
496        realmente não vale o preço           0
497         super produto maravilhoso           1
498    terrivelmente entrega atrasada           0
499  extremamente excelente qualidade           1

[500 rows x 2 columns]


In [72]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["texto"],
    df["sentimento"],
    test_size=0.2,
    random_state=42
)

X_train = X_train.astype("string")
X_test = X_test.astype("string")

In [73]:
from tensorflow.keras.layers import TextVectorization

max_tokens = 1000
sequence_length = 100

vectorizer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(X_train)

vocab = vectorizer.get_vocabulary()

with open("../models/vocab.txt", "w", encoding="utf-8") as f:
    for palavra in vocab:
        f.write(palavra + "\n")

X_train = vectorizer(X_train)
X_test = vectorizer(X_test)

In [74]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

model = Sequential([
    Embedding(
        input_dim=max_tokens,
        output_dim=8
    ),

    GlobalAveragePooling1D(),

    Dense(16, activation="relu"),

    Dense(1, activation="sigmoid")
])

In [75]:
model.compile(
   optimizer="adam",
   loss="binary_crossentropy",
   metrics=["accuracy"]
)

In [76]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    validation_split=0.2
)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.4750 - loss: 0.6947 - val_accuracy: 0.4750 - val_loss: 0.6936
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5063 - loss: 0.6931 - val_accuracy: 0.5250 - val_loss: 0.6928
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5250 - loss: 0.6924 - val_accuracy: 0.5250 - val_loss: 0.6919
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5250 - loss: 0.6917 - val_accuracy: 0.5250 - val_loss: 0.6917
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5250 - loss: 0.6916 - val_accuracy: 0.5250 - val_loss: 0.6916
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5250 - loss: 0.6915 - val_accuracy: 0.5250 - val_loss: 0.6915
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5250 - loss: 0.6914 - val_accuracy: 0.5250 - val_loss: 0.6914
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5250 - loss: 0.6913 - val_accuracy: 0.

In [77]:
loss, accuracy = model.evaluate(X_test, y_test)

print(f"Acurácia: {accuracy:.2f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.2592
Acurácia: 1.00


In [78]:
import pandas as pd

texto = pd.Series(
    ["Esse produto é maravilhoso"],
    dtype="string"
)

texto_vec = vectorizer(texto)

resultado = model.predict(texto_vec)

score = resultado[0][0]

print(f"Probabilidade: {score:.2%}")

if score > 0.5:
    print("Positivo")
else:
    print("Negativo")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
Probabilidade: 66.31%
Positivo
